In [ ]:
from typing import Tuple
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

device = ""
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

def load_data(p: Path | str, train_data_len, val_data_len) -> Tuple[bytes, bytes]:
    with open(p, "r") as fp:
        text = fp.read()
    train_data = text[:train_data_len]
    val_data = text[train_data_len:train_data_len+val_data_len]
    return train_data.encode("utf-8"), val_data.encode("utf-8")


In [ ]:
from collections import Counter

class BPETokenizer:
    def __init__(self):
        self.vocab = [bytes([i]) for i in range(256)]
        self.merges = {}

    def merge_all(self, ids: list[int], pair, new_id):
        i = 0
        res = []
        while i < len(ids):
            if i+1 < len(ids) and (ids[i], ids[i+1]) == pair:
                res.append(new_id)
                i += 2
            else:
                res.append(ids[i])
                i += 1
        return res

    def train(self, data: bytes, vocab_size: int):
        ids = list(data)
        while len(self.vocab) < vocab_size and len(ids) > 1:
            stats = Counter(zip(ids, ids[1:]))
            pair = max(stats, key=lambda p: stats[p])
            new_id = len(self.vocab)
            self.merges[pair] = new_id
            self.vocab.append(self.vocab[pair[0]]+self.vocab[pair[1]])
            ids = self.merge_all(ids, pair, new_id)

    def encode(self, data: bytes):
        ids = list(data)
        while len(ids) > 1:
            pairs = set(zip(ids, ids[1:]))
            p = min(pairs, key=lambda p: self.merges.get(p, 10**9))
            if p not in self.merges:
                break
            ids = self.merge_all(ids, p, self.merges[p])
        return ids

    def decode(self, ids: list[int]):
        return b"".join(map(lambda id: self.vocab[id], ids))


In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, context_len):
        super().__init__()
        self.d_model = d_model
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones((context_len, context_len), dtype=torch.bool)))

    def forward(self, x: torch.Tensor):
        batch_size, seq_length, d_model = x.shape
        K = self.Wk(x)
        Q = self.Wq(x)
        V = self.Wv(x)
        scores = Q @ K.transpose(-1, -2) / (self.d_model**0.5)
        scores = scores.masked_fill(self.mask[:seq_length, :seq_length] == 0, float("-inf"))
        scores = torch.softmax(scores, dim=-1)
        output = scores @ V
        return output

class MLP(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.up_proj = nn.Linear(d_model, 4 * d_model, bias=False)
        self.down_proj = nn.Linear(4 * d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor):
        return self.down_proj(F.gelu(self.up_proj(x)))

class AttentionBlock(nn.Module):
    def __init__(self, d_model: int, context_len: int):
        super().__init__()
        self.attn = Attention(d_model, context_len)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = MLP(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor):
        x = x+self.attn(self.ln1(x))
        x = x+self.ffn(self.ln2(x))
        return x


In [25]:
class Transformer(nn.Module):
    def __init__(self, d_model: int, context_len: int, num_layers: int):
        super().__init__()
        self.d_model = d_model
        self.context_len = context_len
        self.emb = nn.Embedding(len(tokenizer.vocab), d_model)
        self.pos_emb = nn.Embedding(context_len, d_model)
        self.attn = nn.Sequential(*[AttentionBlock(d_model, context_len) for i in range(num_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, len(tokenizer.vocab))
        self.lm_head.weight = self.emb.weight

    def forward(self, x: torch.Tensor):
        seq_len = x.shape[1]
        x = self.emb(x) + self.pos_emb(torch.arange(seq_len, device=x.device))
        x = self.attn(x)
        logits = self.lm_head(self.ln(x))
        return logits


In [54]:
def get_data(data: torch.Tensor, batch_size: int, context_len: int):
    starts = torch.randint(0, data.shape[0]-context_len, (batch_size,))
    offsets = torch.arange(context_len)
    indices = starts[:, None]+offsets
    return data[indices], data[indices+1]

@torch.no_grad
def evaluate(model, val_data, batch_size, context_len, num_batches=20):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_data(val_data, batch_size, context_len)
        logits = model(x)
        log_probs = F.log_softmax(logits, dim=-1)
        losses.append(-log_probs.gather(dim=-1, index=y.unsqueeze(-1)))
        logits = model

    model.train()
    return torch.stack(losses, dim=0).mean()

def train(model, optimizer, steps, batch_size, train_data, val_data, log_steps):
    context_len = model.context_len
    for step in range(steps):
        x, y = get_data(train_data, batch_size, context_len)
        logits = model(x)
        log_probs = F.log_softmax(logits, dim=-1)
        loss = -(log_probs[torch.arange(batch_size)[:, None], torch.arange(context_len), y]).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (step+1) % log_steps == 0 or step == steps-1:
            print(f"step: {step+1}")
            print(f"val loss: {evaluate(model, val_data, batch_size, context_len).item()}")

In [23]:
train_data_raw, val_data_raw = load_data("tiny-shakespeare.txt", 1000000, 100000)
tokenizer = BPETokenizer()
tokenizer.train(train_data_raw, 500)

# print(tokenizer.vocab)
# print(tokenizer.encode(b"for"))
# print(tokenizer.decode([299, 299]))

train_data = torch.tensor(tokenizer.encode(train_data_raw), device=device)
val_data = torch.tensor(tokenizer.encode(val_data_raw), device=device)

1000000


In [55]:
D_MODEL = 128
CONTEXT_LEN = 128
NUM_LAYERS = 2
STEPS = 1000
BATCH_SIZE = 3

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01

model = Transformer(D_MODEL, CONTEXT_LEN, NUM_LAYERS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
train(model, optimizer, STEPS, BATCH_SIZE, train_data, val_data, 50)

step: 50
val loss: 10.518245697021484
step: 100
val loss: 7.041001319885254
step: 150
val loss: 5.958807468414307
step: 200
val loss: 5.606585502624512
step: 250
val loss: 5.4689812660217285
step: 300
val loss: 5.404745578765869
step: 350
val loss: 5.369767665863037
step: 400
val loss: 5.340902805328369
step: 450
val loss: 5.285470485687256
step: 500
val loss: 5.268001079559326
step: 550
val loss: 5.246119022369385
step: 600
val loss: 5.214570045471191
step: 650
val loss: 5.2038187980651855
step: 700
val loss: 5.134575366973877
step: 750
val loss: 5.106264114379883
step: 800
val loss: 5.0658087730407715
step: 850
val loss: 5.029852867126465
step: 900
val loss: 5.010037422180176
step: 950
val loss: 4.954199314117432
step: 1000
val loss: 4.924581050872803
